# Prepping the Pod (Before you get to Jupyter)

Following this:

https://docs.runpod.io/tutorials/pods/run-ollama

1. Log in to your RunPod account and choose + GPU Pod. <br>
2. Choose a GPU Pod like A40.<br>
3. From the availble templates, select the lastet PyTorch template.<br>
4. Select Customize Deployment.<br>
- Add the port 11434 to the list of exposed ports. This port is used by Ollama for HTTP API requests.<br>
- Add the following environment variable to your Pod to allow Ollama to bind to the HTTP port:<br>
- Key: OLLAMA_HOST<br>
- Value: 0.0.0.0<br>
5. Select Set Overrides, Continue, then Deploy.<br>

It is best to put both the container and the volume at whatever size you need. I was running into issues. Just do both to avoid any...

**If you have to adjust the sizes at all, you will lose your downloaded models, the environment setup, and have to repeat the process**

Can run the following in both ssh or Jupyter, but let's stick to SSH then move over to jupyter later.

**apt update**<br>
**apt install lshw -y**

In [1]:
!apt update
!apt install lshw -y

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1581 B]
Get:2 http://archive.ubuntu.com/ubuntu noble InRelease [256 kB]                
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]      
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  Packages [1017 kB]
Get:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]        
Get:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]      
Get:8 http://archive.ubuntu.com/ubuntu noble/multiverse amd64 Packages [331 kB]
Get:9 http://archive.ubuntu.com/ubuntu noble/main amd64 Packages [1808 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [39.8 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble/universe amd64 Packages [19.3 MB]
Get:12 http://archive.ubuntu.com/ubuntu noble/restricted a

Install Ollama Method preferred so one can watch the post/gets in the SSH terminal:

I had this method working for sure in the SSH terminal:
- **curl https://ollama.ai/install.sh | sh**
- **ollama serve**

In [2]:
!curl https://ollama.ai/install.sh | sh

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13281    0 13281    0     0  68767      0 --:--:-- --:--:-- --:--:-- 68813
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


# Ollama Serve MUST BE RAN IN THE SSH CONSOLE, NOT HERE IN THE NOTEBOOK!

Go run "ollama serve" in the ssh

# Start Jupyter Portion - Installs.
For some reason there's an issue with 0.4.0... install the older versions.

In [5]:
!nvidia-smi

Sun Dec 29 16:23:41 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:D2:00.0 Off |                    0 |
|  0%   34C    P8             21W /  300W |       4MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install git+https://github.com/EleutherAI/lm-evaluation-harness.git@main -q


In [4]:
from lm_eval import api

In [5]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)

True
12.8


In [37]:
YAML_mmlu_geo_string = '''
task: demo_mmlu_high_school_geography
dataset_path: cais/mmlu
dataset_name: high_school_geography
description: "The following are multiple choice questions (with answers) about high school geography.\n\n"
test_split: test
fewshot_split: dev
fewshot_config:
  sampler: first_n
output_type: multiple_choice
doc_to_text: "{{question.strip()}}\nA. {{choices[0]}}\nB. {{choices[1]}}\nC. {{choices[2]}}\nD. {{choices[3]}}\nAnswer:"
doc_to_choice: ["A", "B", "C", "D"]
doc_to_target: answer
metric_list:
  - metric: acc
    aggregation: mean
    higher_is_better: true
  - metric: acc_norm
    aggregation: mean
    higher_is_better: true
'''
with open('mmlu_high_school_geography.yaml', 'w') as f:
    f.write(YAML_mmlu_geo_string)


In [45]:
YAML_mmlu_geo_string = '''
task: demo_mmlu_high_school_geography
dataset_path: cais/mmlu
dataset_name: high_school_geography
description: >-
  The following are multiple choice questions (with answers)
  about high school geography.

test_split: test
fewshot_split: dev

fewshot_config:
  sampler: first_n

output_type: multiple_choice

doc_to_text: |
  {{question.strip()}}
  A. {{choices[0]}}
  B. {{choices[1]}}
  C. {{choices[2]}}
  D. {{choices[3]}}
  Answer:

doc_to_choice: ["A", "B", "C", "D"]
doc_to_target: answer

metric_list:
  - metric: acc
    aggregation: mean
    higher_is_better: true

  - metric: acc_norm
    aggregation: mean
    higher_is_better: true
'''
with open('mmlu_high_school_geography.yaml', 'w') as f:
    f.write(YAML_mmlu_geo_string)


In [13]:
! pip install hf_transfer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 50.6 MB/s  0:00:00


In [46]:
!lm_eval \
  --model hf \
  --model_args pretrained=EleutherAI/pythia-2.8b \
  --tasks demo_mmlu_high_school_geography \
  --limit 25 \
  --output output/demo-mmlu \
  --log_samples \
  --device cuda 


2025-11-14:16:10:35 WARNING  [__main__:369]  --limit SHOULD ONLY BE USED FOR TESTING.REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2025-11-14:16:10:35 ERROR    [__main__:419] Tasks were not found: demo_mmlu_high_school_geography
                                               Try `lm-eval --tasks list` for list of available tasks
Traceback (most recent call last):
  File "/usr/local/bin/lm_eval", line 7, in <module>
    sys.exit(cli_evaluate())
             ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/__main__.py", line 423, in cli_evaluate
    raise ValueError(
ValueError: Tasks not found: demo_mmlu_high_school_geography. Try `lm-eval --tasks {list_groups,list_subtasks,list_tags,list}` to list out all available names for task groupings; only (sub)tasks; tags; or all of the above, or pass '--verbosity DEBUG' to troubleshoot task registration issues.


In [54]:
!lm_eval \
  --model hf \
  --model_args pretrained=EleutherAI/pythia-2.8b \
  --include_path ./ \
  --tasks demo_mmlu_high_school_geography \
  --limit 25 \
  --output output/demo-mmlu \
  --log_samples \
  --device cuda 


2025-11-14:16:15:49 INFO     [__main__:348] Including path: ./
Traceback (most recent call last):
  File "/usr/local/bin/lm_eval", line 7, in <module>
    sys.exit(cli_evaluate())
             ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/__main__.py", line 361, in cli_evaluate
    task_manager = TaskManager(include_path=args.include_path, metadata=metadata)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/__init__.py", line 36, in __init__
    self._task_index = self.initialize_tasks(
                       ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/__init__.py", line 83, in initialize_tasks
    tasks = self._get_task_and_group(task_dir)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/__init__.py", line 496, in _get_task_and_group
    config = utils.load_yaml_config

In [48]:
import yaml
with open("mmlu_high_school_geography.yaml", "rb") as f:
    print(yaml.safe_load(f))


{'task': 'demo_mmlu_high_school_geography', 'dataset_path': 'cais/mmlu', 'dataset_name': 'high_school_geography', 'description': 'The following are multiple choice questions (with answers) about high school geography.', 'test_split': 'test', 'fewshot_split': 'dev', 'fewshot_config': {'sampler': 'first_n'}, 'output_type': 'multiple_choice', 'doc_to_text': '{{question.strip()}}\nA. {{choices[0]}}\nB. {{choices[1]}}\nC. {{choices[2]}}\nD. {{choices[3]}}\nAnswer:\n', 'doc_to_choice': ['A', 'B', 'C', 'D'], 'doc_to_target': 'answer', 'metric_list': [{'metric': 'acc', 'aggregation': 'mean', 'higher_is_better': True}, {'metric': 'acc_norm', 'aggregation': 'mean', 'higher_is_better': True}]}


In [50]:
!pwd ls -R .


/workspace


In [41]:
import lm_eval, os, inspect
print(os.path.dirname(inspect.getfile(lm_eval)))


/usr/local/lib/python3.12/dist-packages/lm_eval


In [91]:
import os
import shutil

for root, dirs, files in os.walk(".", topdown=False):
    for d in dirs:
        if d == ".ipynb_checkpoints":
            path = os.path.join(root, d)
            print("Removing:", path)
            shutil.rmtree(path)


Removing: ./.ipynb_checkpoints


# HERE

In [95]:
!lm_eval \
    --model hf \
    --model_args pretrained=EleutherAI/pythia-2.8b \
    --include_path ./tasks \
    --tasks sysengbench \
    --limit 25 \
    --output output/sysengbench \
    --log_samples \
    --device cuda 

2025-11-14:16:52:22 INFO     [__main__:348] Including path: ./tasks
2025-11-14:16:52:33 WARNING  [__main__:369]  --limit SHOULD ONLY BE USED FOR TESTING.REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2025-11-14:16:52:33 INFO     [__main__:450] Selected Tasks: ['sysengbench']
2025-11-14:16:52:33 INFO     [evaluator:202] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-11-14:16:52:33 INFO     [evaluator:240] Initializing hf model, with arguments: {'pretrained': 'EleutherAI/pythia-2.8b'}
2025-11-14:16:52:34 INFO     [models.huggingface:158] Using device 'cuda'
2025-11-14:16:52:34 INFO     [models.huggingface:420] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda'}
`torch_dtype` is deprecated! Use `dtype` instead!
README.md: 100%|█████████████████████████████| 24.0/24.0 [00:00<00:00, 63.6kB/s]
test.csv: 855kB [00:00, 50.0MB/s]
Generating test split: 100%|██████|

In [111]:
# Test List A40
model_list = [
    "phi4-reasoning:14b",       # ~11 GB, 1×A40
    "deepseek-r1:8b",
]

In [101]:
!pip install lm-eval[api] -q

In [103]:
import subprocess
from datetime import datetime

# Generate a unique log file name with a timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
progress_file = f"progress_log_{timestamp}.txt"  # File to log progress

failed_models = []
total_models = len(model_list)  # Total number of models to process

# Create the log file with a header
with open(progress_file, "w") as file:
    file.write(f"Progress Log - Batch Run {timestamp}\n")
    file.write("=================================\n")

# Step 5: Workflow to pull, benchmark, and remove models
for idx, model in enumerate(model_list, start=1):  # Enumerate for progress tracking
    try:
        # Get the current timestamp for processing
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # Debug: Current model being processed
        print(f"\n--- Processing model: {model} ({idx}/{total_models}) at {current_time} ---")
        
        # Log the current model and progress to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Processing model: {model} ({idx}/{total_models})\n")
        
        # Step 3: Run the benchmark using subprocess
        print(f"Step 3: Running benchmark for model: {model}")
        base_url = "http://localhost:11434/v1/chat/completions"  # Ensure Ollama server is running on this URL
        include_path = "./tasks"
        tasks = "sysengbench"
        output_dir = "output/sysengbench/"
        log_samples = True
        batch_size = "auto"
        temperature = 0.0
        apply_chat_template = True

        # Construct the benchmark command dynamically
        log_samples_flag = "--log_samples" if log_samples else ""
        apply_template_flag = "--apply_chat_template" if apply_chat_template else ""

        command = f"""
        lm_eval \
            --model local-chat-completions \
            --model_args model='{model}',base_url='{base_url}',num_concurrent=1 \
            --include_path {include_path} \
            --tasks {tasks} \
            --output {output_dir} \
            {log_samples_flag} \
            --num_fewshot 0 \
            --batch_size {batch_size} \
            --gen_kwargs temperature={temperature} \
            {apply_template_flag}
        """

        # Run the command using subprocess
        print(f"Running benchmark with command:\n{command}")
        subprocess.run(command, shell=True, check=True)

    except Exception as e:
        # Get the current timestamp for the error log
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Debug: Handle any errors that occur
        print(f"An error occurred while processing model {model} at {current_time}: {e}")
        print("Skipping to the next model.\n")

        # Log the failure to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Failed to process model: {model}\n")

        # Add the model to the failed list
        failed_models.append(model)



--- Processing model: phi4-reasoning:14b (1/1) at 2025-11-14 17:06:28 ---
Step 3: Running benchmark for model: phi4-reasoning:14b
Running benchmark with command:

        lm_eval             --model local-chat-completions             --model_args model='phi4-reasoning:14b',base_url='http://localhost:11434/v1/chat/completions',num_concurrent=1             --include_path ./tasks             --tasks sysengbench             --output output/sysengbench/             --log_samples             --num_fewshot 0             --batch_size auto             --gen_kwargs temperature=0.0             --apply_chat_template
        


2025-11-14:17:06:33 INFO     [__main__:348] Including path: ./tasks
2025-11-14:17:06:41 INFO     [__main__:450] Selected Tasks: ['sysengbench']
2025-11-14:17:06:41 INFO     [evaluator:202] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-11-14:17:06:41 WARNING  [evaluator:214] generation_kwargs: {'temperature': 0.0} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2025-11-14:17:06:41 INFO     [evaluator:240] Initializing local-chat-completions model, with arguments: {'model': 'phi4-reasoning:14b', 'base_url':
        'http://localhost:11434/v1/chat/completions', 'num_concurrent': 1}
2025-11-14:17:06:41 WARNING  [models.api_models:160] Automatic batch size is not supported for API models. Defaulting to batch size 1.
2025-11-14:17:06:41 INFO     [models.api_models:172] Using max length 2048 - 1
2025-11-14:17:06:41 INFO     [mo

local-chat-completions (model=phi4-reasoning:14b,base_url=http://localhost:11434/v1/chat/completions,num_concurrent=1), gen_kwargs: (temperature=0.0), limit: None, num_fewshot: 0, batch_size: auto
|   Tasks   |Version|   Filter   |n-shot|  Metric   |   |Value|   |Stderr|
|-----------|------:|------------|-----:|-----------|---|----:|---|-----:|
|sysengbench|      1|strict-match|     0|exact_match|↑  |    0|±  |     0|



In [117]:
import subprocess
from datetime import datetime

# Generate a unique log file name with a timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
progress_file = f"progress_log_{timestamp}.txt"  # File to log progress

failed_models = []
total_models = len(model_list)  # Total number of models to process

# Create the log file with a header
with open(progress_file, "w") as file:
    file.write(f"Progress Log - Batch Run {timestamp}\n")
    file.write("=================================\n")

# Step 5: Workflow to pull, benchmark, and remove models
for idx, model in enumerate(model_list, start=1):  # Enumerate for progress tracking
    try:
        # Get the current timestamp for processing
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # Debug: Current model being processed
        print(f"\n--- Processing model: {model} ({idx}/{total_models}) at {current_time} ---")
        
        # Log the current model and progress to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Processing model: {model} ({idx}/{total_models})\n")
        
        # Step 3: Run the benchmark using subprocess
        print(f"Step 3: Running benchmark for model: {model}")
        base_url = "http://localhost:11434/v1/chat/completions"  # Ensure Ollama server is running on this URL
        include_path = "./tasks"
        tasks = "sysengbench"
        output_dir = "output/sysengbench/"
        log_samples = True
        batch_size = "auto"
        temperature = 0.0
        apply_chat_template = True

        # Construct the benchmark command dynamically
        log_samples_flag = "--log_samples" if log_samples else ""
        apply_template_flag = "--apply_chat_template" if apply_chat_template else ""

        command = f"""
        lm_eval \
            --model local-chat-completions \
            --model_args model='{model}',base_url='{base_url}',num_concurrent=1,hidethinking=true \
            --include_path {include_path} \
            --tasks {tasks} \
            --limit 25 \
            --output {output_dir} \
            {log_samples_flag} \
            --num_fewshot 0 \
            --batch_size {batch_size} \
            --gen_kwargs temperature={temperature} \
            {apply_template_flag}
        """

        # Run the command using subprocess
        print(f"Running benchmark with command:\n{command}")
        subprocess.run(command, shell=True, check=True)

    except Exception as e:
        # Get the current timestamp for the error log
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Debug: Handle any errors that occur
        print(f"An error occurred while processing model {model} at {current_time}: {e}")
        print("Skipping to the next model.\n")

        # Log the failure to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Failed to process model: {model}\n")

        # Add the model to the failed list
        failed_models.append(model)



--- Processing model: phi4-reasoning:14b (1/2) at 2025-11-14 18:13:15 ---
Step 3: Running benchmark for model: phi4-reasoning:14b
Running benchmark with command:

        lm_eval             --model local-chat-completions             --model_args model='phi4-reasoning:14b',base_url='http://localhost:11434/v1/chat/completions',num_concurrent=1,hidethinking=true             --include_path ./tasks             --tasks sysengbench             --limit 25             --output output/sysengbench/             --log_samples             --num_fewshot 0             --batch_size auto             --gen_kwargs temperature=0.0             --apply_chat_template
        


2025-11-14:18:13:20 INFO     [__main__:348] Including path: ./tasks
2025-11-14:18:13:32 WARNING  [__main__:369]  --limit SHOULD ONLY BE USED FOR TESTING.REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2025-11-14:18:13:32 INFO     [__main__:450] Selected Tasks: ['sysengbench']
2025-11-14:18:13:32 INFO     [evaluator:202] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-11-14:18:13:32 WARNING  [evaluator:214] generation_kwargs: {'temperature': 0.0} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2025-11-14:18:13:32 INFO     [evaluator:240] Initializing local-chat-completions model, with arguments: {'model': 'phi4-reasoning:14b', 'base_url':
        'http://localhost:11434/v1/chat/completions', 'num_concurrent': 1, 'hidethinking': True}
2025-11-14:18:13:32 WARNING  [models.api_models:160] Automatic batch size is not supported

local-chat-completions (model=phi4-reasoning:14b,base_url=http://localhost:11434/v1/chat/completions,num_concurrent=1,hidethinking=true), gen_kwargs: (temperature=0.0), limit: 25.0, num_fewshot: 0, batch_size: auto
|   Tasks   |Version|   Filter   |n-shot|  Metric   |   |Value|   |Stderr|
|-----------|------:|------------|-----:|-----------|---|----:|---|-----:|
|sysengbench|      1|strict-match|     0|exact_match|↑  | 0.24|±  |0.0872|


--- Processing model: deepseek-r1:8b (2/2) at 2025-11-14 18:14:42 ---
Step 3: Running benchmark for model: deepseek-r1:8b
Running benchmark with command:

        lm_eval             --model local-chat-completions             --model_args model='deepseek-r1:8b',base_url='http://localhost:11434/v1/chat/completions',num_concurrent=1,hidethinking=true             --include_path ./tasks             --tasks sysengbench             --limit 25             --output output/sysengbench/             --log_samples             --num_fewshot 0             --batch_si

2025-11-14:18:14:47 INFO     [__main__:348] Including path: ./tasks
2025-11-14:18:15:02 WARNING  [__main__:369]  --limit SHOULD ONLY BE USED FOR TESTING.REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2025-11-14:18:15:02 INFO     [__main__:450] Selected Tasks: ['sysengbench']
2025-11-14:18:15:02 INFO     [evaluator:202] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-11-14:18:15:02 WARNING  [evaluator:214] generation_kwargs: {'temperature': 0.0} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2025-11-14:18:15:02 INFO     [evaluator:240] Initializing local-chat-completions model, with arguments: {'model': 'deepseek-r1:8b', 'base_url':
        'http://localhost:11434/v1/chat/completions', 'num_concurrent': 1, 'hidethinking': True}
2025-11-14:18:15:02 WARNING  [models.api_models:160] Automatic batch size is not supported for

local-chat-completions (model=deepseek-r1:8b,base_url=http://localhost:11434/v1/chat/completions,num_concurrent=1,hidethinking=true), gen_kwargs: (temperature=0.0), limit: 25.0, num_fewshot: 0, batch_size: auto
|   Tasks   |Version|   Filter   |n-shot|  Metric   |   |Value|   |Stderr|
|-----------|------:|------------|-----:|-----------|---|----:|---|-----:|
|sysengbench|      1|strict-match|     0|exact_match|↑  |    0|±  |     0|



In [83]:
# Ran before... now not running. 
# !accelerate launch --no_python
!lm_eval \
    --model hf \
    --model_args pretrained=EleutherAI/pythia-2.8b \
    --tasks mmlu_high_school_geography \
    --limit 25 \
    --output output/mmlu \
    --log_samples \
    --device cuda 



2025-11-14:16:44:00 WARNING  [__main__:369]  --limit SHOULD ONLY BE USED FOR TESTING.REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2025-11-14:16:44:00 INFO     [__main__:450] Selected Tasks: ['mmlu_high_school_geography']
2025-11-14:16:44:00 INFO     [evaluator:202] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2025-11-14:16:44:00 INFO     [evaluator:240] Initializing hf model, with arguments: {'pretrained': 'EleutherAI/pythia-2.8b'}
2025-11-14:16:44:00 INFO     [models.huggingface:158] Using device 'cuda'
2025-11-14:16:44:01 INFO     [models.huggingface:420] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda'}
`torch_dtype` is deprecated! Use `dtype` instead!
2025-11-14:16:44:14 INFO     [api.task:434] Building contexts for mmlu_high_school_geography on rank 0...
100%|██████████████████████████████████████████| 25/25 [00:00<00:00, 347.57it/s]
2025-11-14:16:44

In [20]:
# !accelerate launch --no_python
!lm_eval \
    --model hf \
    --model_args pretrained=EleutherAI/pythia-2.8b \
    --include_path ./ \
    --tasks demo_mmlu_high_school_geography \
    --limit 25 \
    --output output/demo-mmlu \
    --log_samples \
    --device cuda \
    --verbosity DEBUG



2025-11-14:15:49:04 INFO     [__main__:348] Including path: ./
Traceback (most recent call last):
  File "/usr/local/bin/lm_eval", line 7, in <module>
    sys.exit(cli_evaluate())
             ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/__main__.py", line 361, in cli_evaluate
    task_manager = TaskManager(include_path=args.include_path, metadata=metadata)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/__init__.py", line 36, in __init__
    self._task_index = self.initialize_tasks(
                       ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/__init__.py", line 83, in initialize_tasks
    tasks = self._get_task_and_group(task_dir)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/__init__.py", line 496, in _get_task_and_group
    config = utils.load_yaml_config

In [79]:
cmd = (
    "lm_eval "
    "--model hf "
    "--model_args pretrained=EleutherAI/pythia-2.8b "
    "--include_path ./ "
    "--tasks demo_mmlu_high_school_geography "
    "--limit 25 "
    "--output output/demo-mmlu "
    "--log_samples "
    "--device cuda "
    "--verbosity DEBUG"
)

!$cmd

2025-11-14:16:36:31 INFO     [__main__:348] Including path: ./
Traceback (most recent call last):
  File "/usr/local/bin/lm_eval", line 7, in <module>
    sys.exit(cli_evaluate())
             ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/__main__.py", line 361, in cli_evaluate
    task_manager = TaskManager(include_path=args.include_path, metadata=metadata)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/__init__.py", line 36, in __init__
    self._task_index = self.initialize_tasks(
                       ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/__init__.py", line 83, in initialize_tasks
    tasks = self._get_task_and_group(task_dir)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/__init__.py", line 496, in _get_task_and_group
    config = utils.load_yaml_config

In [27]:
import os
print(os.getcwd())


/workspace


In [30]:
!ls

mmlu_high_school_geography.yaml  runpod-manual-on-vm.ipynb


In [32]:
!find . -maxdepth 4 -iname "*mmlu_high_school_geography*"

./mmlu_high_school_geography.yaml
./.cache/huggingface/datasets/_workspace_.cache_huggingface_datasets_cais___mmlu_high_school_geography_0.0.0_c30699e8356da336a370243923dbaf21066bb9fe.lock
./.ipynb_checkpoints/mmlu_high_school_geography-checkpoint.yaml


In [35]:
!python - << 'PY'
import lm_eval, os, inspect
pkg_dir = os.path.dirname(inspect.getfile(lm_eval))
print("lm_eval package dir:", pkg_dir)
tasks_dir = os.path.join(pkg_dir, "tasks")
print("Tasks dir candidate:", tasks_dir)
print("Contents of tasks dir:", os.listdir(tasks_dir))



/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
lm_eval package dir: /usr/local/lib/python3.12/dist-packages/lm_eval
Tasks dir candidate: /usr/local/lib/python3.12/dist-packages/lm_eval/tasks
Contents of tasks dir: ['README.md', '__init__.py', 'aclue', 'acpbench', 'aexams', 'afrimgsm', 'afrimmlu', 'afrixnli', 'afrobench', 'agieval', 'aime', 'alghafa', 'anli', 'arab_culture', 'arab_culture_completion', 'arabic_leaderboard_complete', 'arabic_leaderboard_light', 'arabicmmlu', 'aradice', 'arc', 'arc_mt', 'arithmetic', 'asdiv', 'babi', 'babilong', 'bangla', 'basque_bench', 'basqueglue', 'bbh', 'bbq', 'belebele', 'benchmarks', 'bertaqa', 'bhs', 'bigbench', 'blimp', 'blimp_nl', 'c4', 'cabbq', 'careqa', 'catalan_bench', 'ceval', 'chartqa', 'click', 'cmmlu', 'code_x_glue', 'common_voice', 'commonsense_qa', 'copal_id', 'coqa', 'crows_pairs', 'csatqa', 'darija_bench', 'darijahellaswag', 'darijammlu', 'discrim_eval', 'drop', 'egyhellaswag', 'egymmlu', 'eq

NameError: name 'PY' is not defined

In [55]:
YAML_syseng_string = '''
task: sysengbench
dataset_path: rabell/SysEngBench
dataset_name: null
output_type: generate_until
training_split: null
validation_split: null
test_split: test
doc_to_text: "Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Your response must consist solely of the single letter corresponding to the best answer, chosen from one of A, B, C or D.
Any response other than a single letter (A, B, C, or D) will be considered invalid.\n
Question: {{question}}\n
A. {{choiceA}}\n
B. {{choiceB}}\n
C. {{choiceC}}\n
D. {{choiceD}}\n
Answer:"
doc_to_target: "{{answer}}"

generation_kwargs:
  temperature: 0.0
  max_tokens: 20
  until:
    - "</s>"
    - "\n"

filter_list:
  - name: "strict-match"
    filter:
      - function: "regex"
        regex_pattern: "([ABCD])"
      - function: "take_first"
metric_list:
  - metric: exact_match
    aggregation: mean
    higher_is_better: true
    ignore_punctuation: true
    ignore_case: true
metadata:
  version: 1.0
dataset_kwargs:
  trust_remote_code: true
'''
with open('sysengbench.yaml', 'w') as f:
    f.write(YAML_syseng_string)

In [58]:
!lm_eval \
  --model hf \
  --model_args pretrained=EleutherAI/pythia-2.8b \
  --include_path ./ \
  --tasks sysengbench \
  --limit 25 \
  --output output/sysengbench \
  --log_samples \
  --device cuda 


2025-11-14:16:21:48 INFO     [__main__:348] Including path: ./sysengbench.yaml
2025-11-14:16:21:58 WARNING  [__main__:369]  --limit SHOULD ONLY BE USED FOR TESTING.REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2025-11-14:16:21:58 ERROR    [__main__:419] Tasks were not found: sysengbench
                                               Try `lm-eval --tasks list` for list of available tasks
Traceback (most recent call last):
  File "/usr/local/bin/lm_eval", line 7, in <module>
    sys.exit(cli_evaluate())
             ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/__main__.py", line 423, in cli_evaluate
    raise ValueError(
ValueError: Tasks not found: sysengbench. Try `lm-eval --tasks {list_groups,list_subtasks,list_tags,list}` to list out all available names for task groupings; only (sub)tasks; tags; or all of the above, or pass '--verbosity DEBUG' to troubleshoot task registration issues.


In [64]:
import lm_eval.utils as U
import inspect

print(inspect.getsource(U.load_yaml_config))


def debug_load_yaml_config(yaml_path=None, yaml_config=None, yaml_dir=None, mode="full"):
    print("\n====================")
    print("DEBUG: load_yaml_config called")
    print("yaml_path:", yaml_path)

    # Same logic up to the crash
    if mode == "simple":
        constructor_fn = U.ignore_constructor
    elif mode == "full":
        if yaml_path is None:
            raise ValueError("yaml_path must be provided if mode is 'full'.")
        constructor_fn = functools.partial(U.import_function, yaml_path=Path(yaml_path))

    loader = yaml.CLoader if yaml.__with_libyaml__ else yaml.FullLoader
    yaml.add_constructor("!function", constructor_fn, Loader=loader)

    if yaml_config is None:
        try:
            with open(yaml_path, "rb") as file:
                yaml_config = yaml.load(file, Loader=loader)
        except Exception as e:
            print("ERROR: YAML load exception:", e)
            raise

    print("yaml_config type:", type(yaml_config))

    if yaml_dir is Non

In [65]:
import lm_eval.utils as U
import os
import yaml
import functools
from pathlib import Path

_original_load_yaml_config = U.load_yaml_config

def debug_load_yaml_config(yaml_path=None, yaml_config=None, yaml_dir=None, mode="full"):
    print("\n====================")
    print("DEBUG: load_yaml_config called")
    print("yaml_path:", yaml_path)

    # Same logic up to the crash
    if mode == "simple":
        constructor_fn = U.ignore_constructor
    elif mode == "full":
        if yaml_path is None:
            raise ValueError("yaml_path must be provided if mode is 'full'.")
        constructor_fn = functools.partial(U.import_function, yaml_path=Path(yaml_path))

    loader = yaml.CLoader if yaml.__with_libyaml__ else yaml.FullLoader
    yaml.add_constructor("!function", constructor_fn, Loader=loader)

    if yaml_config is None:
        try:
            with open(yaml_path, "rb") as file:
                yaml_config = yaml.load(file, Loader=loader)
        except Exception as e:
            print("ERROR: YAML load exception:", e)
            raise

    print("yaml_config type:", type(yaml_config))

    if yaml_dir is None:
        yaml_dir = os.path.dirname(yaml_path)

    assert yaml_dir is not None

    # If None → this is the bad file
    if yaml_config is None:
        print("❌ ERROR: yaml_config is None. This file is EMPTY or INVALID:", yaml_path)
        raise TypeError("yaml_config is None for path: " + str(yaml_path))

    # If dict → continue to original behavior
    return _original_load_yaml_config(
        yaml_path=yaml_path,
        yaml_config=yaml_config,
        yaml_dir=yaml_dir,
        mode=mode
    )

U.load_yaml_config = debug_load_yaml_config

print("Monkey patch installed.")


Monkey patch installed.


In [63]:
!lm_eval \
    --model hf \
    --model_args pretrained=EleutherAI/pythia-2.8b \
    --include_path /workspace \
    --tasks demo_mmlu_high_school_geography \
    --limit 25 \
    --device cuda \
    --verbosity DEBUG


2025-11-14:16:24:31 INFO     [__main__:348] Including path: /workspace
Traceback (most recent call last):
  File "/usr/local/bin/lm_eval", line 7, in <module>
    sys.exit(cli_evaluate())
             ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/__main__.py", line 361, in cli_evaluate
    task_manager = TaskManager(include_path=args.include_path, metadata=metadata)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/__init__.py", line 36, in __init__
    self._task_index = self.initialize_tasks(
                       ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/__init__.py", line 83, in initialize_tasks
    tasks = self._get_task_and_group(task_dir)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/__init__.py", line 496, in _get_task_and_group
    config = utils.load_yam

In [66]:
import lm_eval.utils as U
import lm_eval.tasks
import yaml
import functools
import os
from pathlib import Path

def debug_load_yaml_config(yaml_path=None, yaml_config=None, yaml_dir=None, mode="full"):
    print("\n====================")
    print("DEBUG: load_yaml_config called")
    print("yaml_path:", yaml_path)

    # same YAML load logic
    if mode == "simple":
        constructor_fn = U.ignore_constructor
    elif mode == "full":
        if yaml_path is None:
            raise ValueError("yaml_path must be provided when mode='full'")
        constructor_fn = functools.partial(U.import_function, yaml_path=Path(yaml_path))

    loader = yaml.CLoader if yaml.__with_libyaml__ else yaml.FullLoader
    yaml.add_constructor("!function", constructor_fn, Loader=loader)

    if yaml_config is None:
        with open(yaml_path, "rb") as f:
            yaml_config = yaml.load(f, Loader=loader)

    print("yaml_config type:", type(yaml_config))

    if yaml_config is None:
        print("❌ BAD FILE:", yaml_path)
        raise TypeError("yaml_config is None for file: " + yaml_path)

    if yaml_dir is None:
        yaml_dir = os.path.dirname(yaml_path)

    return debug_load_yaml_config._orig(
        yaml_path=yaml_path,
        yaml_config=yaml_config,
        yaml_dir=yaml_dir,
        mode=mode
    )

# Attach original
debug_load_yaml_config._orig = U.load_yaml_config

# Replace in utils module
U.load_yaml_config = debug_load_yaml_config

# ALSO replace in tasks module (lm_eval imports it there too!)
import sys
for name, module in sys.modules.items():
    if module and hasattr(module, "load_yaml_config"):
        setattr(module, "load_yaml_config", debug_load_yaml_config)

print("Monkey patch installed everywhere.")


Monkey patch installed everywhere.


In [67]:
import lm_eval.utils as U
print(U.load_yaml_config)


<function debug_load_yaml_config at 0x724321e432e0>


In [69]:
!lm_eval \
  --model hf \
  --model_args pretrained=EleutherAI/pythia-2.8b \
  --include_path ./ \
  --tasks demo_mmlu_high_school_geography \
  --limit 25 \
  --device cuda \
  --verbosity DEBUG


2025-11-14:16:28:14 INFO     [__main__:348] Including path: ./
Traceback (most recent call last):
  File "/usr/local/bin/lm_eval", line 7, in <module>
    sys.exit(cli_evaluate())
             ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/__main__.py", line 361, in cli_evaluate
    task_manager = TaskManager(include_path=args.include_path, metadata=metadata)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/__init__.py", line 36, in __init__
    self._task_index = self.initialize_tasks(
                       ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/__init__.py", line 83, in initialize_tasks
    tasks = self._get_task_and_group(task_dir)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/__init__.py", line 496, in _get_task_and_group
    config = utils.load_yaml_config

In [70]:
import lm_eval.utils as U
import yaml, functools, os
from pathlib import Path
import sys

def debug_load_yaml_config(yaml_path=None, yaml_config=None, yaml_dir=None, mode="full"):
    print("\n====================")
    print("DEBUG: load_yaml_config CALLED")
    print("yaml_path:", yaml_path)

    if mode == "simple":
        constructor_fn = U.ignore_constructor
    elif mode == "full":
        constructor_fn = functools.partial(U.import_function, yaml_path=Path(yaml_path))

    loader = yaml.CLoader if yaml.__with_libyaml__ else yaml.FullLoader
    yaml.add_constructor("!function", constructor_fn, Loader=loader)

    if yaml_config is None:
        with open(yaml_path, "rb") as f:
            yaml_config = yaml.load(f, Loader=loader)

    print("yaml_config type:", type(yaml_config))

    if yaml_config is None:
        print("❌ BAD YAML FILE:", yaml_path)
        raise TypeError("yaml_config is None for: " + yaml_path)

    if yaml_dir is None:
        yaml_dir = os.path.dirname(yaml_path)

    return _orig_func(
        yaml_path=yaml_path,
        yaml_config=yaml_config,
        yaml_dir=yaml_dir,
        mode=mode
    )

# Save original
_orig_func = U.load_yaml_config

# Patch in utils
U.load_yaml_config = debug_load_yaml_config

# Patch everywhere else
patched_count = 0
for name, module in sys.modules.items():
    if hasattr(module, "load_yaml_config"):
        setattr(module, "load_yaml_config", debug_load_yaml_config)
        patched_count += 1

print("Monkey patch installed in", patched_count, "modules.")


Monkey patch installed in 3 modules.


In [71]:
import lm_eval.tasks
print(lm_eval.tasks.load_yaml_config)


AttributeError: module 'lm_eval.tasks' has no attribute 'load_yaml_config'

In [72]:
import lm_eval.tasks as T
import inspect
print(inspect.getsource(T))


import collections
import inspect
import logging
import os
from functools import partial
from typing import Dict, List, Mapping, Optional, Union

from lm_eval import utils
from lm_eval.api.group import ConfigurableGroup, GroupConfig
from lm_eval.api.task import ConfigurableTask, Task
from lm_eval.evaluator_utils import get_subtask_list


GROUP_ONLY_KEYS = list(GroupConfig().to_dict().keys())

eval_logger = logging.getLogger(__name__)


class TaskManager:
    """TaskManager indexes all tasks from the default `lm_eval/tasks/`
    and an optional directory if provided.

    """

    def __init__(
        self,
        verbosity: Optional[str] = None,
        include_path: Optional[Union[str, List]] = None,
        include_defaults: bool = True,
        metadata: Optional[dict] = None,
    ) -> None:
        if verbosity is not None:
            utils.setup_logging(verbosity)
        self.include_path = include_path
        self.metadata = metadata
        self._task_index = self.initializ

In [73]:
import lm_eval.tasks as T
T.load_yaml_config = debug_load_yaml_config
print("Local reference patched")


Local reference patched


In [74]:
import lm_eval.utils as U
import lm_eval.tasks as T

print("utils:", U.load_yaml_config)
print("tasks:", T.load_yaml_config)


utils: <function debug_load_yaml_config at 0x7241c071c5e0>
tasks: <function debug_load_yaml_config at 0x7241c071c5e0>


In [76]:
!lm_eval \
  --model hf \
  --model_args pretrained=EleutherAI/pythia-2.8b \
  --include_path ./ \
  --tasks demo_mmlu_high_school_geography \
  --limit 25 \
  --device cuda \
  --verbosity DEBUG


2025-11-14:16:30:56 INFO     [__main__:348] Including path: ./
Traceback (most recent call last):
  File "/usr/local/bin/lm_eval", line 7, in <module>
    sys.exit(cli_evaluate())
             ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/__main__.py", line 361, in cli_evaluate
    task_manager = TaskManager(include_path=args.include_path, metadata=metadata)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/__init__.py", line 36, in __init__
    self._task_index = self.initialize_tasks(
                       ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/__init__.py", line 83, in initialize_tasks
    tasks = self._get_task_and_group(task_dir)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lm_eval/tasks/__init__.py", line 496, in _get_task_and_group
    config = utils.load_yaml_config

In [78]:
from lm_eval.__main__ import cli_evaluate

args = """
--model hf
--model_args pretrained=EleutherAI/pythia-2.8b
--include_path ./
--tasks demo_mmlu_high_school_geography
--limit 25
--device cuda
--verbosity DEBUG
""".split()

cli_evaluate(args)


AttributeError: 'list' object has no attribute 'wandb_args'

In [82]:
import lm_eval.__main__ as main
import inspect

# 1. Find the parser/parse function
print([name for name in dir(main) if "parse" in name])


['argparse', 'parse_eval_args', 'setup_parser', 'try_parse_json']


In [ ]:
# !pip install ollama==0.4.0 lm_eval lm_eval[api] -q
!pip install lm_eval lm_eval[api] -q


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python -m pip install --upgrade pip


In [7]:
!pip install ollama==0.3.3

  Attempting uninstall: ollama
    Found existing installation: ollama 0.4.0
    Uninstalling ollama-0.4.0:
      Successfully uninstalled ollama-0.4.0

[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python -m pip install --upgrade pip


# Preparing Ollama server for inferencing

## Model array to pull desired models
### [ Preferred Method ] Manual input to arrays

In [ ]:
# Test List A40
model_list = [
    "gpt-oss:20b",              # ~14 GB, 1×A40
    "phi4-reasoning:14b",       # ~11 GB, 1×A40
]

In [ ]:
# Batch 1xA40
# --- Notes for planning disk & GPUs ---
# A40 VRAM per GPU: 48 GB
# Recommended number of A40s (rough rule): ceil(model_size_gb / 48)
# Total disk needed (GB): sum of the per-model sizes you plan to install
#
# Example (pseudo):
#   import math
#   total_gb = sum([14, 4.9, 43, ...])
#   gpus_for_one_model = math.ceil(43 / 48)  # -> 1 for a 43 GB model
#   # Real VRAM depends on batch size, KV cache, server overhead, etc.

# 29 models
model_list = [
    "gpt-oss:20b",              # ~14 GB, 1×A40
    "deepseek-r1:8b",           # ~4.9 GB, 1×A40
    "deepseek-r1:14b",          # ~9.0 GB, 1×A40
    "deepseek-r1:32b",          # ~20 GB, 1×A40
    "deepseek-r1:70b",          # ~43 GB, 1×A40
    "mistral-small3.2:24b",     # ~15 GB, 1×A40
    "magistral:24b",            # ~14 GB, 1×A40
    "devstral:24b",             # ~14 GB, 1×A40
    "phi4-reasoning:14b",       # ~11 GB, 1×A40
    "phi4-reasoning:plus",      # ~11 GB, 1×A40
    "gemma3n:e2b",              # ~? GB (smaller than e4b), 1×A40
    "gemma3n:e4b",              # ~7.5 GB, 1×A40
    "gemma3:270m",              # ~— GB (very small), 1×A40
    "gemma3:1b",                # ~1.3 GB, 1×A40
    "gemma3:4b",                # ~2.6 GB, 1×A40
    "gemma3:12b",               # ~7.8 GB, 1×A40
    "gemma3:27b",               # ~17 GB, 1×A40
    "qwen3:0.6b",               # ~0.40 GB, 1×A40
    "qwen3:1.7b",               # ~1.1 GB, 1×A40
    "qwen3:4b",                 # ~2.6 GB, 1×A40
    "qwen3:8b",                 # ~5.5 GB, 1×A40
    "qwen3:14b",                # ~9.1 GB, 1×A40
    "qwen3:30b",                # ~19 GB, 1×A40
    "qwen3:32b",                # ~20 GB, 1×A40
    "llama3.2:1b",              # ~1.3 GB, 1×A40
    "llama3.2:3b",              # ~2.0 GB, 1×A40
    "llama3.3:70b",             # ~43 GB, 1×A40
    "mistral:7b",               # ~4.4 GB, 1×A40
    "mistral:instruct",         # ~4.1 GB, 1×A40
]



In [ ]:
# Batch 2xA40 or more
model_list = [
    "gpt-oss:120b",             # ~65 GB, 2×A40
    "qwen3:235b",               # ~141 GB, 3×A40
    "llama4:16x17b",            # ~67 GB, 2×A40
]

### Automatic Excel sheet import for model list array

In [10]:
!pip install openpyxl -q


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python -m pip install --upgrade pip


In [61]:
import pandas as pd

# Read the Excel file and select the "ollama-ggufs" sheet
file_path = "20241111 Benchmark Database.xlsx"  # Replace with your actual file path
sheet_name = "ollama-ggufs"

# Load the sheet into a DataFrame
df = pd.read_excel(file_path, sheet_name=sheet_name)

# Step 3a: Filter for models below 48GB
models_below_48GB = df[df["Model Size (GB)"] < 48].sort_values(by="Model Size (GB)", ascending=False)
count_below_48GB = models_below_48GB.shape[0]
print(f"Count of models below 48GB: {count_below_48GB}")
print("Models below 48GB (sorted by size descending):")
print(models_below_48GB[["Ollama Command", "Model Size (GB)"]])

# Step 3b: Filter for models above 48GB
models_above_48GB = df[df["Model Size (GB)"] >= 48].sort_values(by="Model Size (GB)", ascending=False)
count_above_48GB = models_above_48GB.shape[0]
print(f"\nCount of models above or equal to 48GB: {count_above_48GB}")
print("\nModels above 48GB (sorted by size descending):")
print(models_above_48GB[["Ollama Command", "Model Size (GB)"]])

# Select the model list based on the number of GPUs we're using
model_list = models_below_48GB["Ollama Command"].tolist()
# model_list = models_above_48GB["Ollama Command"].tolist()

Count of models below 48GB: 153
Models below 48GB (sorted by size descending):
                      Ollama Command  Model Size (GB)
5         llama3.3:70b-instruct-q4_1           44.000
6       llama3.3:70b-instruct-q4_K_M           43.000
4         llama3.3:70b-instruct-q4_0           40.000
7       llama3.3:70b-instruct-q4_K_S           40.000
146  mixtral:8x7b-instruct-v0.1-q6_K           37.000
..                               ...              ...
18         llama3.2:1b-instruct-q4_0            0.771
15       llama3.2:1b-instruct-q3_K_L            0.733
16       llama3.2:1b-instruct-q3_K_M            0.691
17       llama3.2:1b-instruct-q3_K_S            0.642
14         llama3.2:1b-instruct-q2_K            0.581

[153 rows x 2 columns]

Count of models above or equal to 48GB: 10

Models above 48GB (sorted by size descending):
                      Ollama Command  Model Size (GB)
0         llama3.3:70b-instruct-fp16            141.0
133  mixtral:8x7b-instruct-v0.1-fp16             

## Pulling the Models
### [ Not Preferred Method ] Using ollama python library
Does not pipe the status of the model pull. You have no idea what speed it's pulling the model.

In [7]:
import ollama

In [5]:
# Loop through the models and call ollama.pull
for model in model_list:
    print(f"Attempting to pull: {model}")  # Print before pulling
    try:
        ollama.pull(model)
        print(f"Successfully pulled: {model}")
    except Exception as e:
        print(f"Failed to pull: {model}. Error: {e}")


Attempting to pull: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q8_0
Failed to pull: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q8_0. Error: Server disconnected without sending a response.
Attempting to pull: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q6_K_L
Failed to pull: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q6_K_L. Error: [Errno 111] Connection refused
Attempting to pull: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q6_K
Failed to pull: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q6_K. Error: [Errno 111] Connection refused
Attempting to pull: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q5_K_L
Failed to pull: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q5_K_L. Error: [Errno 111] Connection refused


In [9]:
ollama.list()

{'models': []}

###  Using cmd line commands
Pipe the status of the model pull so you can see how fast models are downloading.

In [ ]:
def pull_models(models):
    for model in models:
        print(f"Pulling model: {model}")
        !ollama pull {model}
        
failed_models = []

for model in model_list:
    try:
        # Debug: Current model being processed
        print(f"\n--- Processing model: {model} ---")

        # Pull the model
        print(f"Step 1: Pulling model: {model}")
        pull_models([model])
        print(f"Model {model} pulled successfully.\n")

    except Exception as e:
        # Debug: Handle any errors that occur
        print(f"An error occurred while processing model {model}: {e}")
        print("Skipping to the next model.\n")

        # Add the model to the failed list
        failed_models.append(model)

### [Preferred Method] Going for parallelization of cmd lines

This can be refined -- it's not reporting back...

In [48]:
import subprocess
import threading
import ipywidgets as widgets
from IPython.display import display
import time

output_widgets = {model: widgets.Textarea(value="Waiting...\n", layout=widgets.Layout(width="100%", height="100px")) 
                 for model in model_list}
start_button = widgets.Button(description="Start Pulling")
status_label = widgets.HTML(value="Status: Ready")

def log(model, message):
    """Thread-safe logging to widgets"""
    current_value = output_widgets[model].value
    output_widgets[model].value = current_value + f"{message}\n"
    # Auto-scroll to bottom
    output_widgets[model].value = output_widgets[model].value

def pull_model(model):
    try:
        log(model, "Starting pull...")
        
        # Run the pull command in a separate subprocess
        process = subprocess.Popen(
            ["ollama", "pull", model],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            universal_newlines=True
        )
        
        # Track if we've seen any output
        had_output = False
        last_output_time = time.time()
        
        while True:
            # Check both stdout and stderr
            outputs = [
                (process.stdout.readline(), ""),
                (process.stderr.readline(), "ERROR: ")
            ]
            
            # Process any available output
            for output, prefix in outputs:
                if output:
                    had_output = True
                    last_output_time = time.time()
                    log(model, f"{prefix}{output.strip()}")
            
            # Check if process has finished
            if process.poll() is not None:
                break
                
            # If no output for 30 seconds, show a message
            if had_output and time.time() - last_output_time > 30:
                log(model, "Still pulling... (no new output for 30 seconds)")
                last_output_time = time.time()
            
            # Small sleep to prevent CPU spinning
            time.sleep(0.1)
        
        # Final status check
        if process.returncode == 0:
            log(model, "✅ Pull completed successfully.")
        else:
            log(model, f"❌ Pull failed with return code {process.returncode}.")
            
    except Exception as e:
        log(model, f"❌ Error: {str(e)}")
        raise e

def start_pulling(_):
    start_button.disabled = True
    status_label.value = "Status: Pulling models..."
    
    # Clear previous output
    for model in model_list:
        output_widgets[model].value = ""
    
    def run_pulls():
        try:
            threads = []
            for model in model_list:
                thread = threading.Thread(target=pull_model, args=(model,))
                threads.append(thread)
                thread.start()
            
            # Wait for all threads to complete
            for thread in threads:
                thread.join()
                
            status_label.value = "Status: All pulls completed"
        except Exception as e:
            status_label.value = f"Status: Error occurred - {str(e)}"
        finally:
            start_button.disabled = False
    
    # Run the pulls in a separate thread to keep UI responsive
    threading.Thread(target=run_pulls).start()

# Bind the start button to the function
start_button.on_click(start_pulling)

# Display the GUI
subviews = [widgets.VBox([widgets.HTML(f"<b>{model}</b>"), output_widgets[model]]) 
            for model in model_list]
display(widgets.VBox([start_button, status_label] + subviews))

Exception in thread Thread-189 (pull_model):
Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
Exception in thread Thread-188 (pull_model):
Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/usr/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_269/3469378465.py", line 70, in pull_model
    self.run()
  File "/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/usr/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_269/3469378465.py", line 70, in pull_model
  File "/tmp/ipykernel_269/3469378465.py", line 24, in pull_

In [ ]:
# trying a more compact reporting mechanism (this code does not currently work properly))
import subprocess
import threading
import ipywidgets as widgets
from IPython.display import display
import time
import re

class CompactModelPuller:
    def __init__(self, model_list):
        self.model_list = model_list
        self.setup_ui()
        
    def setup_ui(self):
        # Create a row for each model with status and progress
        self.rows = {}
        table_rows = []
        
        for model in self.model_list:
            # Create status indicator (⏳ 🔄 ✅ ❌)
            status = widgets.HTML(value="⏳")
            
            # Create progress text
            progress = widgets.HTML(value="Waiting...", layout=widgets.Layout(width='200px'))
            
            # Create progress bar
            bar = widgets.FloatProgress(
                value=0, min=0, max=100,
                description='',
                bar_style='info',
                orientation='horizontal',
                layout=widgets.Layout(width='150px')
            )
            
            # Store widgets for this model
            self.rows[model] = {
                'status': status,
                'progress': progress,
                'bar': bar
            }
            
            # Create a row with model name (shortened), status, progress text, and bar
            short_name = model.split('/')[-1]  # Get just the model name part
            model_label = widgets.HTML(value=f"<code>{short_name}</code>", 
                                    layout=widgets.Layout(width='200px'))
            
            table_rows.append(widgets.HBox([
                model_label, status, progress, bar
            ], layout=widgets.Layout(padding='2px')))
        
        # Create main container
        self.container = widgets.VBox([
            widgets.HTML(value="<h3>Model Downloads</h3>"),
            widgets.Button(description='Start Pulling', 
                         on_click=self.start_pulling,
                         layout=widgets.Layout(width='150px')),
            widgets.VBox(table_rows, 
                        layout=widgets.Layout(border='1px solid #ddd', 
                                            padding='10px',
                                            margin='10px 0'))
        ])
        
    def update_model_status(self, model, status_emoji, progress_text, progress_value=None):
        """Update the status and progress for a model"""
        row = self.rows[model]
        row['status'].value = status_emoji
        row['progress'].value = progress_text
        if progress_value is not None:
            row['bar'].value = progress_value

    def parse_progress(self, line):
        """Parse progress information from ollama output"""
        progress = 0
        if 'downloading' in line.lower():
            match = re.search(r'(\d+)%', line)
            if match:
                progress = int(match.group(1))
        return progress

    def pull_model(self, model):
        try:
            self.update_model_status(model, "🔄", "Starting pull...", 0)
            
            process = subprocess.Popen(
                ["ollama", "pull", model],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                universal_newlines=True
            )
            
            while True:
                outputs = [
                    (process.stdout.readline(), ""),
                    (process.stderr.readline(), "ERROR: ")
                ]
                
                for output, prefix in outputs:
                    if output:
                        # Update progress bar if possible
                        progress = self.parse_progress(output)
                        if progress > 0:
                            self.update_model_status(model, "🔄", output.strip(), progress)
                        else:
                            self.update_model_status(model, "🔄", output.strip())
                
                if process.poll() is not None:
                    break
                    
                time.sleep(0.1)
            
            if process.returncode == 0:
                self.update_model_status(model, "✅", "Complete", 100)
            else:
                self.update_model_status(model, "❌", f"Failed (code {process.returncode})", 0)
                
        except Exception as e:
            self.update_model_status(model, "❌", f"Error: {str(e)}", 0)

    def start_pulling(self, _):
        # Disable the button
        self.container.children[1].disabled = True
        
        # Reset all status indicators
        for model in self.model_list:
            self.update_model_status(model, "⏳", "Waiting...", 0)
        
        def run_pulls():
            try:
                threads = []
                for model in self.model_list:
                    thread = threading.Thread(target=self.pull_model, args=(model,))
                    threads.append(thread)
                    thread.start()
                
                for thread in threads:
                    thread.join()
            finally:
                self.container.children[1].disabled = False
        
        threading.Thread(target=run_pulls).start()
        
    def display(self):
        display(self.container)

puller = CompactModelPuller(model_list)
puller.display()

## Verify models pulled - Comparing current models available in Ollama to model list

In [49]:
import ollama
ollama_data = ollama.list()

In [50]:
# Extract the list of model names
ollama_models_avail = [model['name'] for model in ollama_data['models']]


In [51]:
# ollama_models_avail

In [52]:
# Step 3: Compare model_list with Ollama's available models
if set(model_list) == set(ollama_models_avail):
    print("All models downloaded!")
else:
    missing_models = set(model_list) - set(ollama_models_avail)
    print(f"Missing models: {len(missing_models)}")
    print("Missing models:")
    for model in missing_models:
        print(f"- {model}")

Missing models: 14
Missing models:
- mixtral:8x7b-instruct-v0.1-q4_0
- llama3.3:70b-instruct-q4_K_M
- llama3.3:70b-instruct-q4_1
- gemma2:27b-instruct-q4_1
- gemma2:27b-instruct-q3_K_M
- qwen2.5:32b-instruct-q3_K_L
- qwen2.5:14b-instruct-q3_K_S
- gemma2:27b-instruct-q8_0
- mixtral:8x7b-instruct-v0.1-q4_1
- qwen2.5:32b-instruct
- qwen2.5:32b-instruct-q4_K_M
- llama3.3:70b-instruct-q4_K_S
- qwen2.5:14b-instruct-q8_0
- gemma2:9b-instruct-fp16


# Server API check

In [18]:
import requests

def run_openai_api_chat_test(model, prompt, api_url="http://localhost:11434/v1/chat/completions"):
    """
    Runs an OpenAI-style chat completion API test against the specified model using Ollama's API.

    Args:
        model (str): The model to test (e.g., "llama-2-7b-chat").
        prompt (str): The input prompt to test with the model.
        api_url (str): The endpoint for the Ollama API (default: localhost:11434).

    Returns:
        dict: The response from the API.
    """
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},  # Optional system message
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 100,
        "temperature": 0.7,
        "top_p": 1.0
    }
    try:
        response = requests.post(api_url, json=payload)
        response.raise_for_status()  # Raise an exception for HTTP errors
        return response.json()  # Return the response JSON
    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"API test failed for model {model}: {e}")


In [19]:
##### regular compeltions API WORKS!!!
# Example prompt for testing
test_prompt = "Explain the importance of systems engineering."

# can use below as a subset if i dont want to test ALL of them.
# model_list = [
#     "hf.co/bartowski/Llama-3.2-3B-Instruct-GGUF:Q3_K_L",
#     "hf.co/bartowski/Llama-3.2-3B-Instruct-GGUF:Q4_K_S",
# ]

# Step 5: Workflow to pull, benchmark, and remove models
for model in model_list:
    try:
        # Step 3: Run the OpenAI-style Chat API test
        print(f"Step 4: Running OpenAI CHAT Completions API test for model: {model}")
        response2 = run_openai_api_chat_test(model, test_prompt)
        
        # PRINT GENERATED COMPLETION FROM CHAT COMPLETIONS API
        generated_text2 = response2.get("choices", [{}])[0].get("message", {}).get("content", "No output generated.")
        print(generated_text2)
    
    except Exception as e:
        # Debug: Handle any errors that occur
        print(f"An error occurred while processing model {model}: {e}")
        print("Skipping to the next model.\n")

        # Add the model to the failed list
        failed_models.append(model)


Step 4: Running OpenAI CHAT Completions API test for model: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q8_0
Systems engineering is a crucial discipline that plays a vital role in ensuring the success of complex projects and systems across various industries, including aerospace, defense, healthcare, transportation, and more. The importance of systems engineering can be summarized as follows:

1. **Integrated Approach**: Systems engineering provides an integrated approach to designing, developing, testing, and deploying complex systems. It considers all aspects of a system, from requirements to operations, ensuring that they work together seamlessly.
2. **Risk Management**: By identifying and
Step 4: Running OpenAI CHAT Completions API test for model: hf.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q6_K_L
Systems engineering is a vital discipline that plays a crucial role in developing and managing complex systems, such as aerospace, defense, transportation, healthcare, energy, and

# Benchmark Evaluations
## Template
### Starting Evaluations


In [53]:
### updated version to match the MCQ setup
YAML_syseng_string = '''
task: sysengbench
dataset_path: rabell/SysEngBench
dataset_name: null
output_type: generate_until
training_split: null
validation_split: null
test_split: test
doc_to_text: "Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Your response must consist solely of the single letter corresponding to the best answer, chosen from one of A, B, C or D.
Any response other than a single letter (A, B, C, or D) will be considered invalid.\n
Question: {{question}}\n
A. {{choiceA}}\n
B. {{choiceB}}\n
C. {{choiceC}}\n
D. {{choiceD}}\n
Answer:"
doc_to_target: "{{answer}}"

generation_kwargs:
  temperature: 0.0
  max_tokens: 20
  until:
    - "</s>"
    - "\n"

filter_list:
  - name: "strict-match"
    filter:
      - function: "regex"
        regex_pattern: "([ABCD])"
      - function: "take_first"
metric_list:
  - metric: exact_match
    aggregation: mean
    higher_is_better: true
    ignore_punctuation: true
    ignore_case: true
metadata:
  version: 1.0
dataset_kwargs:
  trust_remote_code: true
'''
with open('sysengbench.yaml', 'w') as f:
    f.write(YAML_syseng_string)

In [ ]:
import subprocess
failed_models = []

# Step 5: Workflow to pull, benchmark, and remove models
for model in model_list:
    try:
        # Debug: Current model being processed
        print(f"\n--- Processing model: {model} ---")

        # Step 3: Run the benchmark using the `!` method
        print(f"Step 3: Running benchmark for model: {model}")
        base_url = "http://localhost:11434/v1/chat/completions"  # Ensure Ollama server is running on this URL
        include_path = "./"
        tasks = "sysengbench"
        # limit = 5 # if want to use, need to add the --limit {limit} \ line below
        output_dir = "output/sysengbench/"
        log_samples = True
        batch_size = "auto"
        temperature = 0.0
        apply_chat_template = True

        # Construct the benchmark command dynamically
        log_samples_flag = "--log_samples" if log_samples else ""
        apply_template_flag = "--apply_chat_template" if apply_chat_template else ""

        # Optionally put the limit varable for debugging
        # --limit {limit} \

        command = f"""
        lm_eval \
            --model local-chat-completions \
            --model_args model='{model}',base_url='{base_url}',num_concurrent=1 \
            --include_path {include_path} \
            --tasks {tasks} \
            --output {output_dir} \
            {log_samples_flag} \
            --num_fewshot 0 \
            --batch_size {batch_size} \
            --gen_kwargs temperature={temperature} \
            {apply_template_flag}
        """
        
        # Run the command using the Jupyter `!` magic
        print(f"Running benchmark with command:\n{command}")
        !{command}

    except Exception as e:
        # Debug: Handle any errors that occur
        print(f"An error occurred while processing model {model}: {e}")
        print("Skipping to the next model.\n")

        # Add the model to the failed list
        failed_models.append(model)

### [Preferred Method ] Perform Evaluation/Benchmarking with lm-eval and include a Timestamped Log File

In [ ]:
model_list = ['mixtral:8x7b-instruct-v0.1-q4_0', 'llama3.3:70b-instruct-q4_K_M', 'llama3.3:70b-instruct-q4_1', 'gemma2:27b-instruct-q3_K_M', 'gemma2:27b-instruct-q4_1', 'qwen2.5:32b-instruct-q3_K_L', 'qwen2.5:14b-instruct-q3_K_S', 'gemma2:27b-instruct-q8_0', 'mixtral:8x7b-instruct-v0.1-q4_1', 'qwen2.5:32b-instruct-q4_K_M', 'qwen2.5:32b-instruct', 'llama3.3:70b-instruct-q4_K_S', 'qwen2.5:14b-instruct-q8_0', 'gemma2:9b-instruct-fp16']

In [ ]:
import subprocess
from datetime import datetime

# Generate a unique log file name with a timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
progress_file = f"progress_log_{timestamp}.txt"  # File to log progress

failed_models = []
total_models = len(model_list)  # Total number of models to process

# Create the log file with a header
with open(progress_file, "w") as file:
    file.write(f"Progress Log - Batch Run {timestamp}\n")
    file.write("=================================\n")

# Step 5: Workflow to pull, benchmark, and remove models
for idx, model in enumerate(model_list, start=1):  # Enumerate for progress tracking
    try:
        # Get the current timestamp for processing
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # Debug: Current model being processed
        print(f"\n--- Processing model: {model} ({idx}/{total_models}) at {current_time} ---")
        
        # Log the current model and progress to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Processing model: {model} ({idx}/{total_models})\n")
        
        # Step 3: Run the benchmark using subprocess
        print(f"Step 3: Running benchmark for model: {model}")
        base_url = "http://localhost:11434/v1/chat/completions"  # Ensure Ollama server is running on this URL
        include_path = "./"
        tasks = "sysengbench"
        output_dir = "output/sysengbench/"
        log_samples = True
        batch_size = "auto"
        temperature = 0.0
        apply_chat_template = True

        # Construct the benchmark command dynamically
        log_samples_flag = "--log_samples" if log_samples else ""
        apply_template_flag = "--apply_chat_template" if apply_chat_template else ""

        command = f"""
        lm_eval \
            --model local-chat-completions \
            --model_args model='{model}',base_url='{base_url}',num_concurrent=1 \
            --include_path {include_path} \
            --tasks {tasks} \
            --output {output_dir} \
            {log_samples_flag} \
            --num_fewshot 0 \
            --batch_size {batch_size} \
            --gen_kwargs temperature={temperature} \
            {apply_template_flag}
        """

        # Run the command using subprocess
        print(f"Running benchmark with command:\n{command}")
        subprocess.run(command, shell=True, check=True)

    except Exception as e:
        # Get the current timestamp for the error log
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Debug: Handle any errors that occur
        print(f"An error occurred while processing model {model} at {current_time}: {e}")
        print("Skipping to the next model.\n")

        # Log the failure to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Failed to process model: {model}\n")

        # Add the model to the failed list
        failed_models.append(model)


# New code

Ollama does not currently expose logits or per-token logprobs through its API. Since lm_eval relies on those values for tasks like loglikelihood and loglikelihood_rolling, these evaluation modes are not supported with Ollama at this time. The only usable interface is 1generate_until1, which supports generative tasks that stop on a given delimiter, hence the usage of the `generate_until` output_type.

## SysEngBench

In [ ]:
YAML_syseng_string = '''
task: sysengbench
dataset_path: rabell/SysEngBench
dataset_name: null
output_type: generate_until
training_split: null
validation_split: null
test_split: test
doc_to_text: "Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Your response must consist solely of the single letter corresponding to the best answer, chosen from one of A, B, C or D.
Any response other than a single letter (A, B, C, or D) will be considered invalid.\n
Question: {{question}}\n
A. {{choiceA}}\n
B. {{choiceB}}\n
C. {{choiceC}}\n
D. {{choiceD}}\n
Answer:"
doc_to_target: "{{answer}}"

generation_kwargs:
  temperature: 0.0
  max_tokens: 20
  until:
    - "</s>"
    - "\n"

filter_list:
  - name: "strict-match"
    filter:
      - function: "regex"
        regex_pattern: "([ABCD])"
      - function: "take_first"
metric_list:
  - metric: exact_match
    aggregation: mean
    higher_is_better: true
    ignore_punctuation: true
    ignore_case: true
metadata:
  version: 1.0
dataset_kwargs:
  trust_remote_code: true
'''
with open('sysengbench.yaml', 'w') as f:
    f.write(YAML_syseng_string)

In [ ]:
import subprocess
from datetime import datetime

# Generate a unique log file name with a timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
progress_file = f"progress_log_{timestamp}.txt"  # File to log progress

failed_models = []
total_models = len(model_list)  # Total number of models to process

# Create the log file with a header
with open(progress_file, "w") as file:
    file.write(f"Progress Log - Batch Run {timestamp}\n")
    file.write("=================================\n")

# Step 5: Workflow to pull, benchmark, and remove models
for idx, model in enumerate(model_list, start=1):  # Enumerate for progress tracking
    try:
        # Get the current timestamp for processing
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # Debug: Current model being processed
        print(f"\n--- Processing model: {model} ({idx}/{total_models}) at {current_time} ---")
        
        # Log the current model and progress to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Processing model: {model} ({idx}/{total_models})\n")
        
        # Step 3: Run the benchmark using subprocess
        print(f"Step 3: Running benchmark for model: {model}")
        base_url = "http://localhost:11434/v1/chat/completions"  # Ensure Ollama server is running on this URL
        include_path = "./"
        tasks = "sysengbench"
        output_dir = "output/sysengbench/"
        log_samples = True
        batch_size = "auto"
        temperature = 0.0
        apply_chat_template = True

        # Construct the benchmark command dynamically
        log_samples_flag = "--log_samples" if log_samples else ""
        apply_template_flag = "--apply_chat_template" if apply_chat_template else ""

        command = f"""
        lm_eval \
            --model local-chat-completions \
            --model_args model='{model}',base_url='{base_url}',num_concurrent=1 \
            --include_path {include_path} \
            --tasks {tasks} \
            --output {output_dir} \
            {log_samples_flag} \
            --num_fewshot 0 \
            --batch_size {batch_size} \
            --gen_kwargs temperature={temperature} \
            {apply_template_flag}
        """

        # Run the command using subprocess
        print(f"Running benchmark with command:\n{command}")
        subprocess.run(command, shell=True, check=True)

    except Exception as e:
        # Get the current timestamp for the error log
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Debug: Handle any errors that occur
        print(f"An error occurred while processing model {model} at {current_time}: {e}")
        print("Skipping to the next model.\n")

        # Log the failure to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Failed to process model: {model}\n")

        # Add the model to the failed list
        failed_models.append(model)


## SysEngBench-A

In [ ]:
YAML_syseng_string = '''
task: sysengbench-a
dataset_path: rabell/SysEngBench-A
dataset_name: null
output_type: generate_until
training_split: null
validation_split: null
test_split: test
doc_to_text: "Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Your response must consist solely of the single letter corresponding to the best answer, chosen from one of A, B, C or D.
Any response other than a single letter (A, B, C, or D) will be considered invalid.\n
Question: {{question}}\n
A. {{choiceA}}\n
B. {{choiceB}}\n
C. {{choiceC}}\n
D. {{choiceD}}\n
Answer:"
doc_to_target: "{{answer}}"

generation_kwargs:
  temperature: 0.0
  max_tokens: 20
  until:
    - "</s>"
    - "\n"

filter_list:
  - name: "strict-match"
    filter:
      - function: "regex"
        regex_pattern: "([ABCD])"
      - function: "take_first"
metric_list:
  - metric: exact_match
    aggregation: mean
    higher_is_better: true
    ignore_punctuation: true
    ignore_case: true
metadata:
  version: 1.0
dataset_kwargs:
  trust_remote_code: true
'''
with open('sysengbench-a.yaml', 'w') as f:
    f.write(YAML_syseng_string)

In [ ]:
import subprocess
from datetime import datetime

# Generate a unique log file name with a timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
progress_file = f"progress_log_{timestamp}.txt"  # File to log progress

failed_models = []
total_models = len(model_list)  # Total number of models to process

# Create the log file with a header
with open(progress_file, "w") as file:
    file.write(f"Progress Log - Batch Run {timestamp}\n")
    file.write("=================================\n")

# Step 5: Workflow to pull, benchmark, and remove models
for idx, model in enumerate(model_list, start=1):  # Enumerate for progress tracking
    try:
        # Get the current timestamp for processing
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # Debug: Current model being processed
        print(f"\n--- Processing model: {model} ({idx}/{total_models}) at {current_time} ---")
        
        # Log the current model and progress to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Processing model: {model} ({idx}/{total_models})\n")
        
        # Step 3: Run the benchmark using subprocess
        print(f"Step 3: Running benchmark for model: {model}")
        base_url = "http://localhost:11434/v1/chat/completions"  # Ensure Ollama server is running on this URL
        include_path = "./"
        tasks = "sysengbench-a"
        output_dir = "output/sysengbench-a/"
        log_samples = True
        batch_size = "auto"
        temperature = 0.0
        apply_chat_template = True

        # Construct the benchmark command dynamically
        log_samples_flag = "--log_samples" if log_samples else ""
        apply_template_flag = "--apply_chat_template" if apply_chat_template else ""

        command = f"""
        lm_eval \
            --model local-chat-completions \
            --model_args model='{model}',base_url='{base_url}',num_concurrent=1 \
            --include_path {include_path} \
            --tasks {tasks} \
            --output {output_dir} \
            {log_samples_flag} \
            --num_fewshot 0 \
            --batch_size {batch_size} \
            --gen_kwargs temperature={temperature} \
            {apply_template_flag}
        """

        # Run the command using subprocess
        print(f"Running benchmark with command:\n{command}")
        subprocess.run(command, shell=True, check=True)

    except Exception as e:
        # Get the current timestamp for the error log
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Debug: Handle any errors that occur
        print(f"An error occurred while processing model {model} at {current_time}: {e}")
        print("Skipping to the next model.\n")

        # Log the failure to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Failed to process model: {model}\n")

        # Add the model to the failed list
        failed_models.append(model)


## SysEngBench-B

In [ ]:
YAML_syseng_string = '''
task: sysengbench-b
dataset_path: rabell/SysEngBench-B
dataset_name: null
output_type: generate_until
training_split: null
validation_split: null
test_split: test
doc_to_text: "Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Your response must consist solely of the single letter corresponding to the best answer, chosen from one of A, B, C or D.
Any response other than a single letter (A, B, C, or D) will be considered invalid.\n
Question: {{question}}\n
A. {{choiceA}}\n
B. {{choiceB}}\n
C. {{choiceC}}\n
D. {{choiceD}}\n
Answer:"
doc_to_target: "{{answer}}"

generation_kwargs:
  temperature: 0.0
  max_tokens: 20
  until:
    - "</s>"
    - "\n"

filter_list:
  - name: "strict-match"
    filter:
      - function: "regex"
        regex_pattern: "([ABCD])"
      - function: "take_first"
metric_list:
  - metric: exact_match
    aggregation: mean
    higher_is_better: true
    ignore_punctuation: true
    ignore_case: true
metadata:
  version: 1.0
dataset_kwargs:
  trust_remote_code: true
'''
with open('sysengbench-b.yaml', 'w') as f:
    f.write(YAML_syseng_string)

In [ ]:
import subprocess
from datetime import datetime

# Generate a unique log file name with a timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
progress_file = f"progress_log_{timestamp}.txt"  # File to log progress

failed_models = []
total_models = len(model_list)  # Total number of models to process

# Create the log file with a header
with open(progress_file, "w") as file:
    file.write(f"Progress Log - Batch Run {timestamp}\n")
    file.write("=================================\n")

# Step 5: Workflow to pull, benchmark, and remove models
for idx, model in enumerate(model_list, start=1):  # Enumerate for progress tracking
    try:
        # Get the current timestamp for processing
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # Debug: Current model being processed
        print(f"\n--- Processing model: {model} ({idx}/{total_models}) at {current_time} ---")
        
        # Log the current model and progress to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Processing model: {model} ({idx}/{total_models})\n")
        
        # Step 3: Run the benchmark using subprocess
        print(f"Step 3: Running benchmark for model: {model}")
        base_url = "http://localhost:11434/v1/chat/completions"  # Ensure Ollama server is running on this URL
        include_path = "./"
        tasks = "sysengbench-b"
        output_dir = "output/sysengbench-b/"
        log_samples = True
        batch_size = "auto"
        temperature = 0.0
        apply_chat_template = True

        # Construct the benchmark command dynamically
        log_samples_flag = "--log_samples" if log_samples else ""
        apply_template_flag = "--apply_chat_template" if apply_chat_template else ""

        command = f"""
        lm_eval \
            --model local-chat-completions \
            --model_args model='{model}',base_url='{base_url}',num_concurrent=1 \
            --include_path {include_path} \
            --tasks {tasks} \
            --output {output_dir} \
            {log_samples_flag} \
            --num_fewshot 0 \
            --batch_size {batch_size} \
            --gen_kwargs temperature={temperature} \
            {apply_template_flag}
        """

        # Run the command using subprocess
        print(f"Running benchmark with command:\n{command}")
        subprocess.run(command, shell=True, check=True)

    except Exception as e:
        # Get the current timestamp for the error log
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Debug: Handle any errors that occur
        print(f"An error occurred while processing model {model} at {current_time}: {e}")
        print("Skipping to the next model.\n")

        # Log the failure to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Failed to process model: {model}\n")

        # Add the model to the failed list
        failed_models.append(model)


## SysEngBench-C

In [ ]:
YAML_syseng_string = '''
task: sysengbench-c
dataset_path: rabell/SysEngBench-C
dataset_name: null
output_type: generate_until
training_split: null
validation_split: null
test_split: test
doc_to_text: "Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Your response must consist solely of the single letter corresponding to the best answer, chosen from one of A, B, C or D.
Any response other than a single letter (A, B, C, or D) will be considered invalid.\n
Question: {{question}}\n
A. {{choiceA}}\n
B. {{choiceB}}\n
C. {{choiceC}}\n
D. {{choiceD}}\n
Answer:"
doc_to_target: "{{answer}}"

generation_kwargs:
  temperature: 0.0
  max_tokens: 20
  until:
    - "</s>"
    - "\n"

filter_list:
  - name: "strict-match"
    filter:
      - function: "regex"
        regex_pattern: "([ABCD])"
      - function: "take_first"
metric_list:
  - metric: exact_match
    aggregation: mean
    higher_is_better: true
    ignore_punctuation: true
    ignore_case: true
metadata:
  version: 1.0
dataset_kwargs:
  trust_remote_code: true
'''
with open('sysengbench-c.yaml', 'w') as f:
    f.write(YAML_syseng_string)

In [ ]:
import subprocess
from datetime import datetime

# Generate a unique log file name with a timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
progress_file = f"progress_log_{timestamp}.txt"  # File to log progress

failed_models = []
total_models = len(model_list)  # Total number of models to process

# Create the log file with a header
with open(progress_file, "w") as file:
    file.write(f"Progress Log - Batch Run {timestamp}\n")
    file.write("=================================\n")

# Step 5: Workflow to pull, benchmark, and remove models
for idx, model in enumerate(model_list, start=1):  # Enumerate for progress tracking
    try:
        # Get the current timestamp for processing
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # Debug: Current model being processed
        print(f"\n--- Processing model: {model} ({idx}/{total_models}) at {current_time} ---")
        
        # Log the current model and progress to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Processing model: {model} ({idx}/{total_models})\n")
        
        # Step 3: Run the benchmark using subprocess
        print(f"Step 3: Running benchmark for model: {model}")
        base_url = "http://localhost:11434/v1/chat/completions"  # Ensure Ollama server is running on this URL
        include_path = "./"
        tasks = "sysengbench-c"
        output_dir = "output/sysengbench-c/"
        log_samples = True
        batch_size = "auto"
        temperature = 0.0
        apply_chat_template = True

        # Construct the benchmark command dynamically
        log_samples_flag = "--log_samples" if log_samples else ""
        apply_template_flag = "--apply_chat_template" if apply_chat_template else ""

        command = f"""
        lm_eval \
            --model local-chat-completions \
            --model_args model='{model}',base_url='{base_url}',num_concurrent=1 \
            --include_path {include_path} \
            --tasks {tasks} \
            --output {output_dir} \
            {log_samples_flag} \
            --num_fewshot 0 \
            --batch_size {batch_size} \
            --gen_kwargs temperature={temperature} \
            {apply_template_flag}
        """

        # Run the command using subprocess
        print(f"Running benchmark with command:\n{command}")
        subprocess.run(command, shell=True, check=True)

    except Exception as e:
        # Get the current timestamp for the error log
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Debug: Handle any errors that occur
        print(f"An error occurred while processing model {model} at {current_time}: {e}")
        print("Skipping to the next model.\n")

        # Log the failure to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Failed to process model: {model}\n")

        # Add the model to the failed list
        failed_models.append(model)


## SysEngBench-D

In [ ]:
YAML_syseng_string = '''
task: sysengbench-d
dataset_path: rabell/SysEngBench-D
dataset_name: null
output_type: generate_until
training_split: null
validation_split: null
test_split: test
doc_to_text: "Given the following question and four candidate answers (A, B, C and D), choose the best answer.
Your response must consist solely of the single letter corresponding to the best answer, chosen from one of A, B, C or D.
Any response other than a single letter (A, B, C, or D) will be considered invalid.\n
Question: {{question}}\n
A. {{choiceA}}\n
B. {{choiceB}}\n
C. {{choiceC}}\n
D. {{choiceD}}\n
Answer:"
doc_to_target: "{{answer}}"

generation_kwargs:
  temperature: 0.0
  max_tokens: 20
  until:
    - "</s>"
    - "\n"

filter_list:
  - name: "strict-match"
    filter:
      - function: "regex"
        regex_pattern: "([ABCD])"
      - function: "take_first"
metric_list:
  - metric: exact_match
    aggregation: mean
    higher_is_better: true
    ignore_punctuation: true
    ignore_case: true
metadata:
  version: 1.0
dataset_kwargs:
  trust_remote_code: true
'''
with open('sysengbench-d.yaml', 'w') as f:
    f.write(YAML_syseng_string)

In [ ]:
import subprocess
from datetime import datetime

# Generate a unique log file name with a timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
progress_file = f"progress_log_{timestamp}.txt"  # File to log progress

failed_models = []
total_models = len(model_list)  # Total number of models to process

# Create the log file with a header
with open(progress_file, "w") as file:
    file.write(f"Progress Log - Batch Run {timestamp}\n")
    file.write("=================================\n")

# Step 5: Workflow to pull, benchmark, and remove models
for idx, model in enumerate(model_list, start=1):  # Enumerate for progress tracking
    try:
        # Get the current timestamp for processing
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # Debug: Current model being processed
        print(f"\n--- Processing model: {model} ({idx}/{total_models}) at {current_time} ---")
        
        # Log the current model and progress to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Processing model: {model} ({idx}/{total_models})\n")
        
        # Step 3: Run the benchmark using subprocess
        print(f"Step 3: Running benchmark for model: {model}")
        base_url = "http://localhost:11434/v1/chat/completions"  # Ensure Ollama server is running on this URL
        include_path = "./"
        tasks = "sysengbench-d"
        output_dir = "output/sysengbench-d/"
        log_samples = True
        batch_size = "auto"
        temperature = 0.0
        apply_chat_template = True

        # Construct the benchmark command dynamically
        log_samples_flag = "--log_samples" if log_samples else ""
        apply_template_flag = "--apply_chat_template" if apply_chat_template else ""

        command = f"""
        lm_eval \
            --model local-chat-completions \
            --model_args model='{model}',base_url='{base_url}',num_concurrent=1 \
            --include_path {include_path} \
            --tasks {tasks} \
            --output {output_dir} \
            {log_samples_flag} \
            --num_fewshot 0 \
            --batch_size {batch_size} \
            --gen_kwargs temperature={temperature} \
            {apply_template_flag}
        """

        # Run the command using subprocess
        print(f"Running benchmark with command:\n{command}")
        subprocess.run(command, shell=True, check=True)

    except Exception as e:
        # Get the current timestamp for the error log
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Debug: Handle any errors that occur
        print(f"An error occurred while processing model {model} at {current_time}: {e}")
        print("Skipping to the next model.\n")

        # Log the failure to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Failed to process model: {model}\n")

        # Add the model to the failed list
        failed_models.append(model)


## SysEngBench-OSQ

 THE PROMPT MUST BE UPDATED BELOW FOR OSQ. THIS IS A PLACEHOLDER. NEED TO REFINE THE PROMPT AND YAML STRING FOR FREE RESPONSE.

In [ ]:
YAML_syseng_string = '''
task: sysengbench

'''
with open('sysengbench-osq.yaml', 'w') as f:
    f.write(YAML_syseng_string)

In [ ]:
import subprocess
from datetime import datetime

# Generate a unique log file name with a timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
progress_file = f"progress_log_{timestamp}.txt"  # File to log progress

failed_models = []
total_models = len(model_list)  # Total number of models to process

# Create the log file with a header
with open(progress_file, "w") as file:
    file.write(f"Progress Log - Batch Run {timestamp}\n")
    file.write("=================================\n")

# Step 5: Workflow to pull, benchmark, and remove models
for idx, model in enumerate(model_list, start=1):  # Enumerate for progress tracking
    try:
        # Get the current timestamp for processing
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # Debug: Current model being processed
        print(f"\n--- Processing model: {model} ({idx}/{total_models}) at {current_time} ---")
        
        # Log the current model and progress to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Processing model: {model} ({idx}/{total_models})\n")
        
        # Step 3: Run the benchmark using subprocess
        print(f"Step 3: Running benchmark for model: {model}")
        base_url = "http://localhost:11434/v1/chat/completions"  # Ensure Ollama server is running on this URL
        include_path = "./"
        tasks = "sysengbench-osq"
        output_dir = "output/sysengbench-osq/"
        log_samples = True
        batch_size = "auto"
        temperature = 0.0
        apply_chat_template = True

        # Construct the benchmark command dynamically
        log_samples_flag = "--log_samples" if log_samples else ""
        apply_template_flag = "--apply_chat_template" if apply_chat_template else ""

        command = f"""
        lm_eval \
            --model local-chat-completions \
            --model_args model='{model}',base_url='{base_url}',num_concurrent=1 \
            --include_path {include_path} \
            --tasks {tasks} \
            --output {output_dir} \
            {log_samples_flag} \
            --num_fewshot 0 \
            --batch_size {batch_size} \
            --gen_kwargs temperature={temperature} \
            {apply_template_flag}
        """

        # Run the command using subprocess
        print(f"Running benchmark with command:\n{command}")
        subprocess.run(command, shell=True, check=True)

    except Exception as e:
        # Get the current timestamp for the error log
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Debug: Handle any errors that occur
        print(f"An error occurred while processing model {model} at {current_time}: {e}")
        print("Skipping to the next model.\n")

        # Log the failure to the file with timestamp
        with open(progress_file, "a") as file:
            file.write(f"[{current_time}] Failed to process model: {model}\n")

        # Add the model to the failed list
        failed_models.append(model)
